# Expected Utility Analysis - Plastic Classification

### Scope

This notebook implements and analyzes a classifier based on **expected utility** for the problem of sorting plastics. 

The goal is to compare different **classical classification algorithms** with our expected utility approach for sorting **ABS**, **HIPS**, **PE**, and **PP** plastics.

### Task 1: Assesing utility

Here is the utility matrix corresponding to the plastic sorting problem. 

\begin{array}{c|cccc}
\text{Act} & \text{ABS} & \text{HIPS} & \text{PE} & \text{PP} \\ \hline
\text{predict ABS} & x_1 & x_2 & x_3 & x_4 \\
\text{predict HIPS} & x_5 & x_6 & x_7 & x_8 \\
\text{predict PE} & x_9 & x_{10} & x_{11} & x_{12} \\
\text{predict PP} & x_{13} & x_{14} & x_{15} & x_{16} \\
\end{array}

For the coefficients we decided to choose:

\begin{array}{c|cccc}
\text{Act} & \text{ABS} & \text{HIPS} & \text{PE} & \text{PP} \\ \hline
\text{predict ABS} & +1 & -1 & -5 & -5 \\
\text{predict HIPS} & -1 & +1 & -5 & -5 \\
\text{predict PE} & -5 & -5 & +1 & -2 \\
\text{predict PP} & -5 & -5 & -2 & +1 \\
\end{array}

- +1 on the diagonal: Correct sorting, the purity is preserved, optimal price/quality.

- ABS ⇔ HIPS = −1: Styrenics are similar, reprocessing is still possible, often corrected via secondary sorting.

- PE ⇔ PP = −2: Mixing degrades properties (MFI, rigidity, thermal resistance), can be sold at lower quality.

- Styrenic ⇔ Polyolefin = −5: Contaminated batch, high risk of defects (delamination, cracking, inclusions), it is often rejected or requires expensive treatment.

### Task 2 Objectives:

- Implement the expected utility model.
- Test various classical classification algorithms. 
- Analyse example showing the expected behaviour of the decision maker.

**1. Imports & Configuration**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

# Configuration pour les graphiques
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Configuration pandas pour l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("Successful imports and configuration")")

✅ Imports et configuration terminés


**2. Parameters & Utility Fonctions**

In [ ]:
# Classes de plastiques dans l'ordre canonique
CLASSES = ["ABS", "HiPS", "PE", "PP"]
IDX = {c:i for i,c in enumerate(CLASSES)}

def build_U_4bins():
    """
    Construit la matrice d'utilité pour une classification en 4 bacs
    selon les valeurs spécifiées par l'utilisateur
    
    Actions possibles:
    - classify_as_ABS: classer comme ABS
    - classify_as_HiPS: classer comme HiPS  
    - classify_as_PE: classer comme PE
    - classify_as_PP: classer comme PP
    
    États: [ABS, HiPS, PE, PP]
    
    Justification des valeurs:
    - +1 sur la diagonale: tri correct → pureté conservée, prix/qualité optimaux
    - ABS↔HiPS = -1: styréniques proches → revalorisation encore possible, correction souvent viable par tri secondaire
    - PE↔PP = -2: mélange dégrade les specs (MFI, rigidité, tenue thermique), vendable en qualité inférieure
    - Styrénique ↔ Polyoléfine = -5: lot contaminé, risques de défauts (décollement, fissuration, inclusions) → souvent rebu, traitement coûteux
    
    Returns:
    - U: matrice d'utilité (4,4) où U[action, état] = utilité
    """
    # Matrice selon vos spécifications
    U = np.array([
        [+1, -1, -5, -5],  # Prédire ABS:  correct=+1, erreur HiPS=-1, erreur PE/PP=-5
        [-1, +1, -5, -5],  # Prédire HiPS: erreur ABS=-1, correct=+1, erreur PE/PP=-5
        [-5, -5, +1, -2],  # Prédire PE:   erreur ABS/HiPS=-5, correct=+1, erreur PP=-2
        [-5, -5, -2, +1]   # Prédire PP:   erreur ABS/HiPS=-5, erreur PE=-2, correct=+1
    ], dtype=float)
    
    return U

def expected_utility_4bins(P, U):
    """
    Calcule l'utilité espérée pour chaque action de classification
    
    Parameters:
    - P: (n,4) probabilités par classe dans l'ordre CLASSES
    - U: (4,4) matrice d'utilité 4 bacs
    
    Returns:
    - EU: utilités espérées pour chaque action (n,4)
    - best_action_idx: indices des meilleures actions (n,)
    - best_action: noms des meilleures actions (n,)
    - EU_max: utilités espérées maximales (n,)
    """
    EU = P @ U.T  # (n,4)
    best_action_idx = EU.argmax(axis=1)
    best_action = np.array(CLASSES)[best_action_idx]
    return EU, best_action_idx, best_action, EU.max(axis=1)

def classification_threshold_analysis():
    """
    Analyse des seuils de décision pour la classification 4 bacs
    avec la matrice d'utilité spécifiée
    """
    U = build_U_4bins()
    return U

# Affichage de la matrice d'utilité personnalisée
U_4bins = build_U_4bins()
print("Matrice d'utilité pour classification en 4 bacs (valeurs personnalisées):")
print(f"{'Action \\ État':<15} {'ABS':<8} {'HiPS':<8} {'PE':<8} {'PP':<8}")
for i, action in enumerate(['ABS', 'HiPS', 'PE', 'PP']):
    print(f"{'Prédire ' + action:<15} {U_4bins[i,0]:<8.0f} {U_4bins[i,1]:<8.0f} {U_4bins[i,2]:<8.0f} {U_4bins[i,3]:<8.0f}")


classification_threshold_analysis()

Matrice d'utilité pour classification en 4 bacs (valeurs personnalisées):
Action \ État   ABS      HiPS     PE       PP      
Prédire ABS     1        -1       -5       -5      
Prédire HiPS    -1       1        -5       -5      
Prédire PE      -5       -5       1        -2      
Prédire PP      -5       -5       -2       1       
\nJustification des valeurs:
✅ +1: Classification correcte → pureté optimale
⚠️  -1: Erreur ABS↔HiPS → revalorisation possible
⚠️  -2: Erreur PE↔PP → qualité dégradée mais vendable
❌ -5: Erreur Styrénique↔Polyoléfine → contamination grave
Analyse des seuils de décision:
Pour maximiser l'utilité espérée, une classe i sera choisie quand:
EU_i = Σ P[j] * U[i,j] > EU_k pour tout k≠i
\nSeuils caractéristiques:
- ABS vs HiPS: différence d'utilité = 2 (quand erreur mutuelle)
- PE vs PP: différence d'utilité = 1 (quand erreur mutuelle)
- Styréniques vs Polyoléfines: différence d'utilité = 6 (quand erreur croisée)


array([[ 1., -1., -5., -5.],
       [-1.,  1., -5., -5.],
       [-5., -5.,  1., -2.],
       [-5., -5., -2.,  1.]])

## 3. Chargement et Exploration des Données

In [3]:
# Chargement du dataset PlasticsTrain.csv
print("Chargement du dataset PlasticsTrain.csv...")
df = pd.read_csv("PlasticsTrain.csv", sep=";", decimal=",")

print(f"Forme du dataset: {df.shape}")
print(f"\nColonnes: {list(df.columns)[:10]}... (première 10)")
print(f"\nAperçu des données:")
df.head()

Chargement du dataset PlasticsTrain.csv...
Forme du dataset: (10254, 158)

Colonnes: ['3687cm-1', '3665cm-1', '3642cm-1', '3620cm-1', '3598cm-1', '3577cm-1', '3555cm-1', '3534cm-1', '3514cm-1', '3493cm-1']... (première 10)

Aperçu des données:
Forme du dataset: (10254, 158)

Colonnes: ['3687cm-1', '3665cm-1', '3642cm-1', '3620cm-1', '3598cm-1', '3577cm-1', '3555cm-1', '3534cm-1', '3514cm-1', '3493cm-1']... (première 10)

Aperçu des données:


,3687cm-1,3665cm-1,3642cm-1,3620cm-1,3598cm-1,3577cm-1,3555cm-1,3534cm-1,3514cm-1,3493cm-1,3473cm-1,3453cm-1,3433cm-1,3413cm-1,3394cm-1,3375cm-1,3356cm-1,3337cm-1,3318cm-1,3300cm-1,3282cm-1,3264cm-1,3246cm-1,3229cm-1,3211cm-1,3194cm-1,3177cm-1,3160cm-1,3144cm-1,3127cm-1,3111cm-1,3095cm-1,3079cm-1,3063cm-1,3047cm-1,3032cm-1,3017cm-1,3001cm-1,2986cm-1,2972cm-1,2957cm-1,2942cm-1,2928cm-1,2914cm-1,2899cm-1,2885cm-1,2872cm-1,2858cm-1,2844cm-1,2831cm-1,2817cm-1,2804cm-1,2791cm-1,2778cm-1,2765cm-1,2752cm-1,2740cm-1,2727cm-1,2715cm-1,2703cm-1,2690cm-1,2678cm-1,2666cm-1,2655cm-1,2643cm-1,2631cm-1,2620cm-1,2608cm-1,2597cm-1,2586cm-1,2574cm-1,2563cm-1,2552cm-1,2542cm-1,2531cm-1,2520cm-1,2509cm-1,2499cm-1,2489cm-1,2478cm-1,2468cm-1,2458cm-1,2448cm-1,2438cm-1,2428cm-1,2418cm-1,2408cm-1,2399cm-1,2389cm-1,2379cm-1,2370cm-1,2361cm-1,2351cm-1,2342cm-1,2333cm-1,2324cm-1,2315cm-1,2306cm-1,2297cm-1,2288cm-1,2280cm-1,2271cm-1,2262cm-1,2254cm-1,2245cm-1,2237cm-1,2229cm-1,2220cm-1,2212cm-1,2204cm-1,2196cm-1,2188cm-1,2180cm-1,2172cm-1,2164cm-1,2156cm-1,2148cm-1,2141cm-1,2133cm-1,2125cm-1,2118cm-1,2110cm-1,2103cm-1,2096cm-1,2088cm-1,2081cm-1,2074cm-1,2067cm-1,2059cm-1,2052cm-1,2045cm-1,2038cm-1,2031cm-1,2025cm-1,2018cm-1,2011cm-1,2004cm-1,1997cm-1,1991cm-1,1984cm-1,1978cm-1,1971cm-1,1965cm-1,1958cm-1,1952cm-1,1945cm-1,1939cm-1,1933cm-1,1927cm-1,1920cm-1,1914cm-1,1908cm-1,1902cm-1,1896cm-1,class,line,column,object
0,-1.9962,-1.9648,-1.9881,-1.9980,-1.9971,-1.9819,-1.9819,-1.9361,-1.9541,-1.9066,-1.8644,-1.8680,-1.8250,-1.7587,-1.7488,-1.7139,-1.6529,-1.5830,-1.5149,-1.4261,-1.3670,-1.3051,-1.2290,-1.1106,-1.0318,-0.9583,-0.8794,-0.7682,-0.6759,-0.5908,-0.4698,-0.4250,-0.3891,-0.3326,-0.2403,-0.2152,-0.1856,-0.1184,-0.1408,-0.1579,-0.1354,-0.0611,0.0223,0.2016,0.3468,0.4014,0.4454,0.4534,0.6085,0.7259,0.8003,0.8362,0.8541,0.8657,0.9195,0.8899,0.9652,1.0432,1.0369,1.1006,1.0907,1.1705,1.1929,1.2296,1.2243,1.2108,1.2287,1.2010,1.2305,1.1893,1.2028,1.2072,1.2368,1.2431,1.2592,1.2485,1.2583,1.2395,1.2834,1.2476,1.2350,1.2180,1.2386,1.1499,1.1499,1.1454,1.1149,1.1355,1.0558,0.5735,-0.2296,-1.0506,-1.2567,-1.1151,-1.1026,-0.9789,-0.7360,-0.2323,0.1962,0.5431,0.7429,0.7250,0.8003,0.7842,0.7797,0.8371,0.8003,0.7510,0.7205,0.6910,0.6730,0.6551,0.6013,0.5852,0.5735,0.5655,0.5431,0.5108,0.4481,0.4104,0.3647,0.3306,0.2795,0.2016,0.1657,0.1128,0.1039,0.1541,0.1379,0.1612,0.1487,0.0877,0.0626,0.0824,-0.0162,-0.0826,-0.1077,-0.1399,-0.2206,-0.2161,-0.2457,-0.2027,-0.2717,-0.3344,-0.3165,-0.4402,-0.5343,-0.5442,-0.5908,-0.7198,-0.7871,-0.7629,-0.7754,-0.8005,ABS,1,1,ABS_EMA_noir_1_1_22
1,-1.9320,-1.9329,-1.9521,-2.0063,-1.9845,-1.9687,-1.9670,-1.9364,-1.9101,-1.8629,-1.8340,-1.8611,-1.8078,-1.7728,-1.7483,-1.7247,-1.6678,-1.5883,-1.5060,-1.4457,-1.3608,-1.3040,-1.2095,-1.1308,-1.0495,-0.9550,-0.8614,-0.7591,-0.6681,-0.6121,-0.5089,-0.4381,-0.4031,-0.3349,-0.2561,-0.2395,-0.2334,-0.1932,-0.2037,-0.2107,-0.1958,-0.1652,-0.0681,0.1103,0.2678,0.3238,0.3483,0.3815,0.5258,0.6736,0.7541,0.8179,0.8302,0.8923,0.9107,0.9421,0.9754,1.0768,1.0576,1.1346,1.1250,1.1801,1.2273,1.2317,1.2675,1.2588,1.2649,1.2483,1.2605,1.2089,1.2290,1.2168,1.2562,1.2422,1.2544,1.2824,1.2684,1.3086,1.2920,1.2841,1.2693,1.2115,1.2500,1.1547,1.1608,1.1774,1.0847,1.1285,1.0637,0.6054,-0.2037,-1.0468,-1.2480,-1.1054,-1.0705,-0.9944,-0.7407,-0.2360,0.2249,0.5389,0.7226,0.7637,0.7917,0.7690,0.7891,0.8276,0.7978,0.7593,0.7314,0.6911,0.6763,0.6194,0.5704,0.5914,0.5547,0.5407,0.5328,0.4926,0.4663,0.4174,0.3570,0.3264,0.2485,0.2275,0.1628,0.1235,0.1331,0.1383,0.1226,0.1488,0.1515,0.1182,0.0596,0.1051,0.0010,-0.0803,-0.1022,-0.1337,-0.1870,-0.2168,-0.2421,-0.2063,-0.2745,-0.3305,-0.2972,-0.4197,-0.5527,-0.5579,-0.5649,-0.6917,-0.7661,-0.7538,-0.7529,-0.7827,ABS,1,2,ABS_EMA_noir_1_1_22
2,-1.9331,-1.9322,-1.9736,-1.9463,-1.9832,-1.9656,-1.9489,-1.9269,-1.9040,-1.8714,-1.8477,-1.8494,-1.8160,-1.7860,-1.7561,-1.7420,-1.6830,-1.6012,-1.5703,-1.4480,-1.3731,-1.3071,-1.2428,-1.1266,-1.0562,-0.9514,-0.8660,-0.7472,-0.672

In [4]:
# Extraction des features spectrales
feature_cols = [col for col in df.columns if col not in ['class', 'line', 'column', 'object']]
print(f"Nombre de features spectrales: {len(feature_cols)}")
print(f"Plage spectrale: {feature_cols[0]} à {feature_cols[-1]}")

# Préparation des données
X = df[feature_cols].values
y = df['class'].values

print(f"\nForme des données: X={X.shape}, y={y.shape}")
print(f"Classes disponibles: {np.unique(y)}")

Nombre de features spectrales: 154
Plage spectrale: 3687cm-1 à 1896cm-1

Forme des données: X=(10254, 154), y=(10254,)
Classes disponibles: ['ABS' 'HiPS' 'PE' 'PP']


In [6]:
# Division en ensembles d'entraînement et de validation
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Ensemble d'entraînement: {X_tr.shape[0]} échantillons")
print(f"Ensemble de validation: {X_va.shape[0]} échantillons")

# Vérification de la stratification
train_dist = pd.Series(y_tr).value_counts(normalize=True).sort_index()
val_dist = pd.Series(y_va).value_counts(normalize=True).sort_index()

stratification_df = pd.DataFrame({
    'Train': train_dist,
    'Validation': val_dist
})
print("\nVérification de la stratification:")
print(stratification_df.round(3))

Ensemble d'entraînement: 7177 échantillons
Ensemble de validation: 3077 échantillons

Vérification de la stratification:
      Train  Validation
ABS   0.191       0.191
HiPS  0.286       0.286
PE    0.285       0.285
PP    0.238       0.238


## 4. Implémentation du Classificateur d'Utilité Espérée

### Task 2a: Implémentation du modèle d'utilité espérée

In [7]:
class ExpectedUtilityClassifier(BaseEstimator, ClassifierMixin):
    """
    Classificateur basé sur l'utilité espérée.
    
    Ce classificateur combine un modèle de classification classique 
    avec une matrice d'utilité pour prendre des décisions optimales
    selon le critère d'utilité espérée maximale.
    
    Parameters:
    -----------
    base_classifier : estimator
        Le classificateur de base qui fournit les probabilités P(classe|x)
    utility_matrix : array-like, shape (n_actions, n_classes)
        Matrice d'utilité U[a,s] = utilité de l'action a dans l'état s
    classes : array-like
        Liste des classes dans l'ordre de la matrice d'utilité
    """
    
    def __init__(self, base_classifier, utility_matrix, classes=None):
        self.base_classifier = base_classifier
        self.utility_matrix = np.array(utility_matrix)
        self.classes = classes if classes is not None else CLASSES
        self.classes_ = np.array(self.classes)
        
    def fit(self, X, y):
        """Entraîne le classificateur de base"""
        self.base_classifier.fit(X, y)
        return self
    
    def predict_proba(self, X):
        """Retourne les probabilités par classe"""
        if hasattr(self.base_classifier, "predict_proba"):
            proba = self.base_classifier.predict_proba(X)
            # Réordonner selon self.classes
            base_classes = list(self.base_classifier.classes_)
            reorder_idx = [base_classes.index(c) for c in self.classes]
            return proba[:, reorder_idx]
        else:
            raise ValueError("Le classificateur de base doit avoir predict_proba")
    
    def predict_utility(self, X):
        """
        Calcule l'utilité espérée pour chaque action
        
        Returns:
        --------
        EU : array, shape (n_samples, n_actions)
            Utilité espérée pour chaque action
        """
        P = self.predict_proba(X)  # (n_samples, n_classes)
        EU = P @ self.utility_matrix.T  # (n_samples, n_actions)
        return EU
    
    def predict(self, X):
        """
        Prédit l'action optimale selon le critère EU maximale
        
        Pour le problème de tri des plastiques:
        - action 0: send_styrenics
        - action 1: reject
        """
        EU = self.predict_utility(X)
        best_action_idx = EU.argmax(axis=1)
        # Convertir en labels d'action
        action_labels = ["send_styrenics", "reject"]
        return np.array([action_labels[i] for i in best_action_idx])
    
    def predict_class_by_eu(self, X):
        """
        Version alternative: prédit la classe qui maximise l'utilité espérée
        
        Pour chaque échantillon, trouve la classe qui, si elle était vraie,
        donnerait la meilleure utilité espérée pour l'action optimale.
        """
        P = self.predict_proba(X)  # (n_samples, n_classes)
        EU = self.predict_utility(X)  # (n_samples, n_actions)
        
        # Pour chaque échantillon, trouve l'action optimale
        best_actions = EU.argmax(axis=1)  # (n_samples,)
        
        # Calcule l'utilité de chaque classe pour l'action optimale
        class_utilities = []
        for i, best_action in enumerate(best_actions):
            # Utilité de chaque classe pour cette action optimale
            class_util = self.utility_matrix[best_action, :] * P[i, :]
            class_utilities.append(class_util)
        
        class_utilities = np.array(class_utilities)  # (n_samples, n_classes)
        best_class_idx = class_utilities.argmax(axis=1)
        
        return np.array([self.classes[i] for i in best_class_idx])

print("✅ Classe ExpectedUtilityClassifier implémentée")

✅ Classe ExpectedUtilityClassifier implémentée


## 5. Fonction de Comparaison des Classificateurs

### Task 2b: Test avec plusieurs algorithmes de classification classiques

In [8]:
def compare_classifiers(base_classifiers, X_train, y_train, X_val, y_val, 
                       utility_params={'g': 1.0, 'c': 5.0, 'm': 0.25}):
    """
    Compare les classificateurs classiques avec les classificateurs EU
    
    Returns:
    --------
    results : dict
        Résultats de comparaison pour chaque classificateur
    """
    g, c, m = utility_params['g'], utility_params['c'], utility_params['m']
    U = build_U_caseB(g, c, m)
    
    results = {}
    
    for name, base_clf in base_classifiers:
        print(f"\n--- Évaluation: {name} ---")
        
        # 1. Classificateur classique
        base_clf.fit(X_train, y_train)
        y_pred_classic = base_clf.predict(X_val)
        acc_classic = accuracy_score(y_val, y_pred_classic)
        
        # 2. Classificateur EU
        eu_clf = ExpectedUtilityClassifier(base_clf, U, CLASSES)
        eu_clf.fit(X_train, y_train)  # Déjà entraîné mais pour la forme
        
        # Prédictions EU (actions)
        actions_pred = eu_clf.predict(X_val)
        
        # Prédictions EU (classes)
        y_pred_eu = eu_clf.predict_class_by_eu(X_val)
        acc_eu = accuracy_score(y_val, y_pred_eu)
        
        # Utilités espérées
        EU = eu_clf.predict_utility(X_val)
        avg_eu = EU.max(axis=1).mean()
        
        # Taux d'envoi de styréniques
        send_rate = (actions_pred == "send_styrenics").mean()
        
        results[name] = {
            'classic_accuracy': acc_classic,
            'eu_accuracy': acc_eu,
            'avg_expected_utility': avg_eu,
            'send_rate': send_rate,
            'eu_classifier': eu_clf,
            'classic_predictions': y_pred_classic,
            'eu_predictions': y_pred_eu,
            'actions': actions_pred,
            'utilities': EU
        }
        
        print(f"  Précision classique: {acc_classic:.3f}")
        print(f"  Précision EU: {acc_eu:.3f}")
        print(f"  Utilité espérée moy.: {avg_eu:.3f}")
        print(f"  Taux d'envoi styréniques: {send_rate:.3f}")
    
    return results

print("✅ Fonction de comparaison définie")

✅ Fonction de comparaison définie


In [17]:
# Définition des classificateurs de base à tester
base_classifiers = [
    ("LogReg", LogisticRegression(max_iter=500)),
    ("SVM-RBF", SVC(kernel="rbf", probability=True)),
    ("kNN-15", KNeighborsClassifier(n_neighbors=15)),
    ("RF-300", RandomForestClassifier(n_estimators=300, random_state=42)),
    ("DecisionTree", DecisionTreeClassifier(random_state=42, max_depth=10))
]

# Application de la standardisation pour les modèles qui en ont besoin
standardized_classifiers = []
for name, clf in base_classifiers:
    if name in ["LogReg", "SVM-RBF", "kNN-15"]:
        standardized_classifiers.append((name, make_pipeline(StandardScaler(), clf)))
    else:
        standardized_classifiers.append((name, clf))

print("Classificateurs à tester:")
for name, _ in standardized_classifiers:
    print(f"  - {name}")

Classificateurs à tester:
  - LogReg
  - SVM-RBF
  - kNN-15
  - RF-300
  - DecisionTree


In [18]:
# Comparaison des classificateurs avec système 4-bacs et matrice personnalisée
print("="*80)
print("COMPARAISON DES CLASSIFICATEURS - SYSTÈME 4 BACS")
print("="*80)

# Utilisation de la nouvelle fonction pour 4 bacs avec votre matrice personnalisée
results = compare_classifiers_4bins(standardized_classifiers, X_tr, y_tr, X_va, y_va)

COMPARAISON DES CLASSIFICATEURS - SYSTÈME 4 BACS
\n--- Évaluation: LogReg ---
  Précision classique: 0.917
  Précision EU: 0.918
  Utilité espérée moy.: 0.594
  Distribution EU: {'ABS': 0.197, 'HiPS': 0.283, 'PE': 0.29, 'PP': 0.23}
\n--- Évaluation: SVM-RBF ---
  Précision classique: 0.917
  Précision EU: 0.918
  Utilité espérée moy.: 0.594
  Distribution EU: {'ABS': 0.197, 'HiPS': 0.283, 'PE': 0.29, 'PP': 0.23}
\n--- Évaluation: SVM-RBF ---
  Précision classique: 0.906
  Précision EU: 0.912
  Utilité espérée moy.: 0.576
  Distribution EU: {'ABS': 0.191, 'HiPS': 0.305, 'PE': 0.272, 'PP': 0.232}
\n--- Évaluation: kNN-15 ---
  Précision classique: 0.906
  Précision EU: 0.912
  Utilité espérée moy.: 0.576
  Distribution EU: {'ABS': 0.191, 'HiPS': 0.305, 'PE': 0.272, 'PP': 0.232}
\n--- Évaluation: kNN-15 ---
  Précision classique: 0.893
  Précision EU: 0.889
  Utilité espérée moy.: 0.358
  Distribution EU: {'ABS': 0.183, 'HiPS': 0.296, 'PE': 0.292, 'PP': 0.229}
\n--- Évaluation: RF-300 ---

## 6. Analyse des Résultats

In [ ]:
# Résumé comparatif
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Classic_Accuracy': res['classic_accuracy'],
        'EU_Accuracy': res['eu_accuracy'],
        'Accuracy_Diff': res['eu_accuracy'] - res['classic_accuracy'],
        'Avg_Expected_Utility': res['avg_expected_utility'],
        'Send_Rate': res['send_rate']
    }
    for name, res in results.items()
])

comparison_df = comparison_df.sort_values('Avg_Expected_Utility', ascending=False)

print("RÉSUMÉ COMPARATIF: CLASSIQUE vs EXPECTED UTILITY")
print("="*60)
print(comparison_df.round(4))

print(f"\n🏆 Meilleur modèle selon l'utilité espérée: {comparison_df.iloc[0]['Model']}")
print(f"   Utilité espérée: {comparison_df.iloc[0]['Avg_Expected_Utility']:.4f}")

RÉSUMÉ COMPARATIF: CLASSIQUE vs EXPECTED UTILITY
     Model  Classic_Accuracy  EU_Accuracy  Accuracy_Diff  \
0   LogReg            0.9171       0.6841        -0.2330   
1  SVM-RBF            0.9058       0.6750        -0.2307   
2   kNN-15            0.8928       0.3838        -0.5089   
4     GBDT            0.9100       0.6617        -0.2483   
3   RF-300            0.9236       0.5496        -0.3741   

   Avg_Expected_Utility  Send_Rate  
0                0.3609     0.4186  
1                0.3067     0.4300  
2                0.2907     0.3640  
4                0.2856     0.3942  
3                0.2268     0.3627  

🏆 Meilleur modèle selon l'utilité espérée: LogReg
   Utilité espérée: 0.3609


## 7. Analyse Détaillée des Décisions EU

### Task 2c: Exemples illustrant le comportement attendu du décideur

In [ ]:
def analyze_decision_examples(results, X_val, y_val, n_examples=10):
    """
    Analyse des exemples où le classificateur EU prend des décisions 
    différentes du classificateur classique, illustrant le comportement
    attendu du décideur.
    """
    print("\n" + "="*80)
    print("ANALYSE DES EXEMPLES DE DÉCISIONS EU vs CLASSIQUE")
    print("="*80)
    
    # Prendre le meilleur modèle selon l'utilité espérée
    best_model = max(results.items(), key=lambda x: x[1]['avg_expected_utility'])
    model_name, model_results = best_model
    
    print(f"\nAnalyse basée sur le modèle: {model_name}")
    print(f"Utilité espérée moyenne: {model_results['avg_expected_utility']:.3f}")
    
    eu_clf = model_results['eu_classifier']
    classic_pred = model_results['classic_predictions']
    eu_pred = model_results['eu_predictions']
    actions = model_results['actions']
    utilities = model_results['utilities']
    
    # Calculer les probabilités
    P = eu_clf.predict_proba(X_val)
    
    # Créer un DataFrame pour l'analyse
    analysis_df = pd.DataFrame({
        'y_true': y_val,
        'classic_pred': classic_pred,
        'eu_pred': eu_pred,
        'action': actions,
        'P_ABS': P[:, 0],
        'P_HiPS': P[:, 1], 
        'P_PE': P[:, 2],
        'P_PP': P[:, 3],
        'EU_send': utilities[:, 0],
        'EU_reject': utilities[:, 1],
        'EU_diff': utilities[:, 0] - utilities[:, 1]  # Différence send - reject
    })
    
    print(f"\nMatrice d'utilité utilisée:")
    U = eu_clf.utility_matrix
    print(f"{'Action':<15} {'ABS':<8} {'HiPS':<8} {'PE':<8} {'PP':<8}")
    print(f"{'Send styréniques':<15} {U[0,0]:<8.1f} {U[0,1]:<8.1f} {U[0,2]:<8.1f} {U[0,3]:<8.1f}")
    print(f"{'Reject':<15} {U[1,0]:<8.1f} {U[1,1]:<8.1f} {U[1,2]:<8.1f} {U[1,3]:<8.1f}")
    
    return analysis_df, model_name

# Analyse des exemples
analysis_df, best_model_name = analyze_decision_examples(results, X_va, y_va, n_examples=5)


ANALYSE DES EXEMPLES DE DÉCISIONS EU vs CLASSIQUE

Analyse basée sur le modèle: LogReg
Utilité espérée moyenne: 0.361

Matrice d'utilité utilisée:
Action          ABS      HiPS     PE       PP      
Send styréniques 1.0      1.0      -5.0     -5.0    
Reject          -0.2     -0.2     0.0      0.0     


In [ ]:
# 1. Exemples de rejet prudent
print("\n1. EXEMPLES DE REJET PRUDENT")
print("   (EU choisit 'reject' malgré probabilité styrénique significative)")
print("-" * 70)

# Échantillons avec P(styrénique) > 0.3 mais action = reject
p_styrenic = analysis_df['P_ABS'] + analysis_df['P_HiPS']
cautious_reject = (p_styrenic > 0.3) & (analysis_df['action'] == "reject")

if cautious_reject.sum() > 0:
    examples = analysis_df[cautious_reject].head(3)
    
    for idx, (_, row) in enumerate(examples.iterrows()):
        print(f"Exemple {idx+1}:")
        print(f"  Vraie classe: {row['y_true']}")
        p_sty = row['P_ABS'] + row['P_HiPS']
        print(f"  P(ABS)={row['P_ABS']:.3f}, P(HiPS)={row['P_HiPS']:.3f}, P(styrénique)={p_sty:.3f}")
        print(f"  EU(send)={row['EU_send']:.3f}, EU(reject)={row['EU_reject']:.3f}")
        print(f"  Action EU: {row['action']} (vs classique: {row['classic_pred']})")
        print(f"  → Décision prudente: évite le coût élevé d'une erreur PE/PP envoyé")
        print()
else:
    print("  Aucun exemple trouvé dans cette catégorie.")


1. EXEMPLES DE REJET PRUDENT
   (EU choisit 'reject' malgré probabilité styrénique significative)
----------------------------------------------------------------------
Exemple 1:
  Vraie classe: HiPS
  P(ABS)=0.128, P(HiPS)=0.408, P(styrénique)=0.536
  EU(send)=-1.783, EU(reject)=-0.134
  Action EU: reject (vs classique: HiPS)
  → Décision prudente: évite le coût élevé d'une erreur PE/PP envoyé

Exemple 2:
  Vraie classe: PP
  P(ABS)=0.446, P(HiPS)=0.095, P(styrénique)=0.541
  EU(send)=-1.756, EU(reject)=-0.135
  Action EU: reject (vs classique: ABS)
  → Décision prudente: évite le coût élevé d'une erreur PE/PP envoyé

Exemple 3:
  Vraie classe: PE
  P(ABS)=0.153, P(HiPS)=0.518, P(styrénique)=0.671
  EU(send)=-0.976, EU(reject)=-0.168
  Action EU: reject (vs classique: HiPS)
  → Décision prudente: évite le coût élevé d'une erreur PE/PP envoyé



In [ ]:
# 2. Analyse des comportements par classe
print("\n2. COMPORTEMENT PAR CLASSE VRAIE")
print("-" * 70)

behavior_by_class = []
for classe in CLASSES:
    mask = analysis_df['y_true'] == classe
    if mask.sum() > 0:
        reject_rate = (analysis_df.loc[mask, 'action'] == "reject").mean()
        avg_eu_send = analysis_df.loc[mask, 'EU_send'].mean()
        avg_eu_reject = analysis_df.loc[mask, 'EU_reject'].mean()
        
        behavior_by_class.append({
            'Classe': classe,
            'Taux_Rejet': reject_rate,
            'EU_Send_Moyen': avg_eu_send,
            'EU_Reject_Moyen': avg_eu_reject,
            'Nb_Echantillons': mask.sum()
        })
        
        print(f"{classe}: {reject_rate:.1%} de rejet ({mask.sum()} échantillons)")

behavior_df = pd.DataFrame(behavior_by_class)
print("\nTableau récapitulatif:")
print(behavior_df.round(3))


2. COMPORTEMENT PAR CLASSE VRAIE
----------------------------------------------------------------------
ABS: 14.3% de rejet (588 échantillons)
HiPS: 12.6% de rejet (881 échantillons)
PE: 99.2% de rejet (877 échantillons)
PP: 99.0% de rejet (731 échantillons)

Tableau récapitulatif:
  Classe  Taux_Rejet  EU_Send_Moyen  EU_Reject_Moyen  Nb_Echantillons
0    ABS       0.143          0.560           -0.232              588
1   HiPS       0.126          0.589           -0.233              881
2     PE       0.992         -4.659           -0.014              877
3     PP       0.990         -4.687           -0.013              731


In [ ]:
# 4. Validation du comportement attendu du décideur
print("\n4. VALIDATION DU COMPORTEMENT ATTENDU DU DÉCIDEUR")
print("-" * 70)

print("Comportements conformes aux attentes du DM:")
print(f"✅ Rejet plus fréquent pour PE/PP (coût élevé d'erreur -5.0)")
pe_reject = behavior_df[behavior_df['Classe'] == 'PE']['Taux_Rejet'].iloc[0]
pp_reject = behavior_df[behavior_df['Classe'] == 'PP']['Taux_Rejet'].iloc[0]
print(f"   PE: {pe_reject:.1%}, PP: {pp_reject:.1%}")

print(f"✅ Envoi plus fréquent pour ABS/HiPS (gain potentiel +1.0)")
abs_reject = behavior_df[behavior_df['Classe'] == 'ABS']['Taux_Rejet'].iloc[0]
hips_reject = behavior_df[behavior_df['Classe'] == 'HiPS']['Taux_Rejet'].iloc[0]
print(f"   ABS: {1-abs_reject:.1%} d'envoi, HiPS: {1-hips_reject:.1%} d'envoi")

print(f"✅ Prise en compte de l'incertitude dans les décisions")
print(f"   Le classificateur EU évite les risques élevés même avec probabilité modérée")

# Comparaison avec le seuil Bayésien
threshold = bayes_threshold()
above_threshold = p_styrenic_full > threshold
send_actions = analysis_df['action'] == 'send_styrenics'

print(f"\n📊 Analyse du seuil Bayésien (p_S* = {threshold:.3f}):")
print(f"   Échantillons au-dessus du seuil: {above_threshold.sum()}/{len(analysis_df)} ({above_threshold.mean():.1%})")
print(f"   Actions 'send' effectuées: {send_actions.sum()}/{len(analysis_df)} ({send_actions.mean():.1%})")

# Cohérence avec le seuil
correct_threshold_decisions = (
    (above_threshold & send_actions) | 
    (~above_threshold & ~send_actions)
).mean()
print(f"   Cohérence avec seuil Bayésien: {correct_threshold_decisions:.1%}")


4. VALIDATION DU COMPORTEMENT ATTENDU DU DÉCIDEUR
----------------------------------------------------------------------
Comportements conformes aux attentes du DM:
✅ Rejet plus fréquent pour PE/PP (coût élevé d'erreur -5.0)
   PE: 99.2%, PP: 99.0%
✅ Envoi plus fréquent pour ABS/HiPS (gain potentiel +1.0)
   ABS: 85.7% d'envoi, HiPS: 87.4% d'envoi
✅ Prise en compte de l'incertitude dans les décisions
   Le classificateur EU évite les risques élevés même avec probabilité modérée

📊 Analyse du seuil Bayésien (p_S* = 0.800):
   Échantillons au-dessus du seuil: 1288/3077 (41.9%)
   Actions 'send' effectuées: 1288/3077 (41.9%)
   Cohérence avec seuil Bayésien: 100.0%


## 8. Exemple d'Utilisation Pratique

In [ ]:
# Exemple d'utilisation pratique du classificateur EU
print("EXEMPLE D'UTILISATION PRATIQUE")
print("="*60)

# Prendre le meilleur modèle
best_eu_classifier = results[best_model_name]['eu_classifier']

# Prédictions sur quelques exemples
n_examples_demo = 10
X_demo = X_va[:n_examples_demo]
y_demo = y_va[:n_examples_demo]

print(f"\nPrédictions du classificateur EU optimal ({best_model_name}) sur {n_examples_demo} exemples:")
print("-" * 80)

# Probabilités et décisions
P_demo = best_eu_classifier.predict_proba(X_demo)
EU_demo = best_eu_classifier.predict_utility(X_demo)
actions_demo = best_eu_classifier.predict(X_demo)
classes_demo = best_eu_classifier.predict_class_by_eu(X_demo)

demo_results = []
for i in range(n_examples_demo):
    demo_results.append({
        'Exemple': i+1,
        'Vraie_Classe': y_demo[i],
        'P_ABS': P_demo[i,0],
        'P_HiPS': P_demo[i,1],
        'P_PE': P_demo[i,2],
        'P_PP': P_demo[i,3],
        'EU_Send': EU_demo[i,0],
        'EU_Reject': EU_demo[i,1],
        'Action': actions_demo[i],
        'Classe_Predite': classes_demo[i]
    })

demo_df = pd.DataFrame(demo_results)
print(demo_df.round(3))

EXEMPLE D'UTILISATION PRATIQUE

Prédictions du classificateur EU optimal (LogReg) sur 10 exemples:
--------------------------------------------------------------------------------
   Exemple Vraie_Classe  P_ABS  P_HiPS   P_PE   P_PP  EU_Send  EU_Reject  \
0        1         HiPS  0.128   0.408  0.231  0.232   -1.783     -0.134   
1        2           PE  0.002   0.000  0.997  0.001   -4.987     -0.001   
2        3          ABS  0.853   0.038  0.058  0.052    0.343     -0.223   
3        4           PP  0.000   0.000  0.000  1.000   -5.000     -0.000   
4        5          ABS  1.000   0.000  0.000  0.000    1.000     -0.250   
5        6         HiPS  0.118   0.861  0.018  0.003    0.874     -0.245   
6        7         HiPS  0.000   0.999  0.000  0.000    0.997     -0.250   
7        8         HiPS  0.006   0.994  0.000  0.000    0.998     -0.250   
8        9           PE  0.000   0.000  1.000  0.000   -5.000     -0.000   
9       10          ABS  1.000   0.000  0.000  0.000    1.00

## 9. Conclusions

### Résumé de la Task 2

**a. Implémentation du modèle d'utilité espérée ✅**
- Classe `ExpectedUtilityClassifier` développée avec succès
- Intégration d'un classificateur de base avec une matrice d'utilité
- Calcul optimal des utilités espérées EU = P @ U^T

**b. Tests avec plusieurs algorithmes classiques ✅**
- 5 algorithmes testés: LogReg, SVM-RBF, kNN-15, RF-300, GBDT
- Régression Logistique identifiée comme meilleur modèle (EU = 0.361)
- Comparaison systématique des performances

**c. Analyse du comportement du décideur ✅**
- Rejet prudent: 99% pour PE/PP (évite coût -5.0)
- Envoi stratégique: 85-87% pour ABS/HiPS (vise gain +1.0)
- Prise en compte optimale de l'incertitude

### Points clés

1. **Cohérence théorique**: Le classificateur EU respecte les principes de décision sous incertitude
2. **Comportement rationnel**: Decisions alignées avec les préférences du décideur
3. **Performance pratique**: Utilité espérée maximisée tout en maintenant une précision acceptable
4. **Robustesse**: Résultats cohérents across différents algorithmes de base

Le classificateur d'utilité espérée démontre sa capacité à prendre des décisions optimales dans le contexte du tri des plastiques, en équilibrant intelligemment les gains potentiels et les risques selon les préférences du décideur.

# Task 3: Evidential k-Nearest Neighbourhood (EKNN) Classifier

## Objectifs de la Task 3

Le même problème que la Task 2 mais en utilisant un classificateur EKNN (Evidential k-Nearest Neighbourhood):

- **a.** Implémenter le classificateur EKNN
- **b. & c.** Implémenter le modèle d'utilité espérée avec le classificateur EKNN
- **d.** Mettre en évidence des exemples où l'algorithme k-NN classique donne des résultats différents de ceux d'EKNN

### Concept d'EKNN

L'EKNN est une extension du k-NN classique basée sur la **théorie de l'évidence de Dempster-Shafer**:

1. **Masses de croyance**: Chaque voisin contribue avec une masse de croyance proportionnelle à sa distance
2. **Gestion de l'incertitude**: Permet de quantifier l'ignorance et l'incertitude
3. **Combinaison d'évidences**: Utilise la règle de Dempster pour combiner les évidences des voisins
4. **Probabilités pignistiques**: Conversion des masses de croyance en probabilités pour la prise de décision

Cette approche est particulièrement adaptée aux situations avec incertitude élevée, comme notre problème de classification de plastiques.

## 10. Implémentation du Classificateur EKNN

### Task 3a: Implémentation du classificateur EKNN

In [13]:
class ExpectedUtilityClassifier(BaseEstimator, ClassifierMixin):
    """
    Classificateur basé sur l'utilité espérée pour classification en 4 bacs.
    
    Ce classificateur combine un modèle de classification classique 
    avec une matrice d'utilité pour prendre des décisions optimales
    selon le critère d'utilité espérée maximale.
    
    Parameters:
    -----------
    base_classifier : estimator
        Le classificateur de base qui fournit les probabilités P(classe|x)
    utility_matrix : array-like, shape (n_actions, n_classes)
        Matrice d'utilité U[a,s] = utilité de l'action a dans l'état s
        Pour 4 bacs: U[i,j] = utilité de classer en i quand la vraie classe est j
    classes : array-like
        Liste des classes dans l'ordre de la matrice d'utilité
    """
    
    def __init__(self, base_classifier, utility_matrix, classes=None):
        self.base_classifier = base_classifier
        self.utility_matrix = np.array(utility_matrix)
        self.classes = classes if classes is not None else CLASSES
        self.classes_ = np.array(self.classes)
        
    def fit(self, X, y):
        """Entraîne le classificateur de base"""
        self.base_classifier.fit(X, y)
        return self
    
    def predict_proba(self, X):
        """Retourne les probabilités par classe"""
        if hasattr(self.base_classifier, "predict_proba"):
            proba = self.base_classifier.predict_proba(X)
            # Réordonner selon self.classes
            base_classes = list(self.base_classifier.classes_)
            reorder_idx = [base_classes.index(c) for c in self.classes]
            return proba[:, reorder_idx]
        else:
            raise ValueError("Le classificateur de base doit avoir predict_proba")
    
    def predict_utility(self, X):
        """
        Calcule l'utilité espérée pour chaque action de classification
        
        Returns:
        --------
        EU : array, shape (n_samples, n_actions)
            Utilité espérée pour chaque action (classification en chaque classe)
        """
        P = self.predict_proba(X)  # (n_samples, n_classes)
        EU = P @ self.utility_matrix.T  # (n_samples, n_actions)
        return EU
    
    def predict(self, X):
        """
        Prédit la classe optimale selon le critère EU maximale
        
        Returns:
        --------
        predictions : array
            Classes prédites (indices convertis en noms de classes)
        """
        EU = self.predict_utility(X)
        best_action_idx = EU.argmax(axis=1)
        # Pour classification 4 bacs, l'action = classe de destination
        return np.array([self.classes[i] for i in best_action_idx])
    
    def predict_class_by_eu(self, X):
        """
        Alias pour predict() - prédit la classe qui maximise l'utilité espérée
        """
        return self.predict(X)

print("✅ Classe ExpectedUtilityClassifier pour 4 bacs implémentée")

✅ Classe ExpectedUtilityClassifier pour 4 bacs implémentée


### Test de l'implémentation EKNN

In [14]:
# Nouvelle fonction de comparaison pour classification en 4 bacs avec votre matrice
def compare_classifiers_4bins(base_classifiers, X_train, y_train, X_val, y_val):
    """
    Compare les classificateurs classiques avec les classificateurs EU pour 4 bacs
    en utilisant la matrice d'utilité personnalisée
    
    Returns:
    --------
    results : dict
        Résultats de comparaison pour chaque classificateur
    """
    U = build_U_4bins()  # Utilise votre matrice personnalisée
    
    results = {}
    
    for name, base_clf in base_classifiers:
        print(f"\\n--- Évaluation: {name} ---")
        
        # 1. Classificateur classique
        base_clf.fit(X_train, y_train)
        y_pred_classic = base_clf.predict(X_val)
        acc_classic = accuracy_score(y_val, y_pred_classic)
        
        # 2. Classificateur EU pour 4 bacs
        eu_clf = ExpectedUtilityClassifier(base_clf, U, CLASSES)
        eu_clf.fit(X_train, y_train)  # Déjà entraîné mais pour la forme
        
        # Prédictions EU (classes directement)
        y_pred_eu = eu_clf.predict(X_val)
        acc_eu = accuracy_score(y_val, y_pred_eu)
        
        # Utilités espérées
        EU = eu_clf.predict_utility(X_val)
        avg_eu = EU.max(axis=1).mean()
        
        # Distribution des classifications EU
        eu_distribution = pd.Series(y_pred_eu).value_counts(normalize=True).sort_index()
        
        results[name] = {
            'classic_accuracy': acc_classic,
            'eu_accuracy': acc_eu,
            'avg_expected_utility': avg_eu,
            'eu_distribution': eu_distribution,
            'eu_classifier': eu_clf,
            'classic_predictions': y_pred_classic,
            'eu_predictions': y_pred_eu,
            'utilities': EU
        }
        
        print(f"  Précision classique: {acc_classic:.3f}")
        print(f"  Précision EU: {acc_eu:.3f}")
        print(f"  Utilité espérée moy.: {avg_eu:.3f}")
        print(f"  Distribution EU: {dict(eu_distribution.round(3))}")
    
    return results

# Test rapide de l'implémentation avec votre matrice personnalisée
print("Test de l'implémentation Classification 4 bacs avec matrice personnalisée:")
print("=" * 70)

# Créer un petit échantillon pour tester
np.random.seed(42)
n_test = 100
test_indices = np.random.choice(len(X_tr), n_test, replace=False)
X_test_small = X_tr[test_indices]
y_test_small = y_tr[test_indices]

# Test avec un classificateur simple
test_classifier = LogisticRegression(max_iter=500)

# Préparation pipeline avec standardisation
test_pipeline = make_pipeline(StandardScaler(), test_classifier)

# Test sur échantillon réduit avec votre matrice
test_results = compare_classifiers_4bins(
    [("LogReg-Test", test_pipeline)], 
    X_test_small, y_test_small, 
    X_test_small[:20], y_test_small[:20]  # Validation très réduite pour test
)

print("\\n✅ Tests Classification 4 bacs avec matrice personnalisée terminés avec succès!")
print("\\n📊 Matrice utilisée - Résumé des principes:")
print("   • Gains corrects: +1 (classifications exactes)")
print("   • Erreurs légères: -1 (ABS↔HiPS), -2 (PE↔PP)")  
print("   • Erreurs graves: -5 (contamination croisée styrénique↔polyoléfine)")

Test de l'implémentation Classification 4 bacs avec matrice personnalisée:
\n--- Évaluation: LogReg-Test ---
  Précision classique: 1.000
  Précision EU: 0.950
  Utilité espérée moy.: 0.371
  Distribution EU: {'ABS': 0.15, 'HiPS': 0.4, 'PE': 0.3, 'PP': 0.15}
\n✅ Tests Classification 4 bacs avec matrice personnalisée terminés avec succès!
\n📊 Matrice utilisée - Résumé des principes:
   • Gains corrects: +1 (classifications exactes)
   • Erreurs légères: -1 (ABS↔HiPS), -2 (PE↔PP)
   • Erreurs graves: -5 (contamination croisée styrénique↔polyoléfine)


## 11. EKNN avec Utilité Espérée

### Tasks 3b & 3c: Implémentation du modèle d'utilité espérée avec EKNN

In [ ]:
# Version simplifiée de la comparaison k-NN vs EKNN avec matrice 4x4
def evaluate_knn_variants():
    """Évaluation simplifiée des variantes k-NN et EKNN pour classification 4 bacs"""
    
    print("ÉVALUATION k-NN vs EKNN pour Classification 4 Bacs")
    print("=" * 50)
    
    # Configuratíon simplifiée
    classifiers = [
        ("k-NN (k=5)", KNeighborsClassifier(n_neighbors=5)),
        ("k-NN (k=10)", KNeighborsClassifier(n_neighbors=10)),
        ("EKNN (k=5)", EvidentialKNN(k=5, alpha=0.95, gamma=0.5)),
        ("EKNN (k=10)", EvidentialKNN(k=10, alpha=0.9, gamma=0.3))
    ]
    
    # Standardisation
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    
    # Matrice d'utilité 4x4
    U = build_U_4bins(correct_gain=10.0, wrong_cost=5.0)
    results = {}
    
    for name, clf in classifiers:
        # Entraînement
        clf.fit(X_tr_scaled, y_tr)
        
        # Prédictions et probabilités
        y_pred = clf.predict(X_va_scaled)
        P = clf.predict_proba(X_va_scaled)
        
        # Ajustement de l'ordre des classes si nécessaire
        if hasattr(clf, 'classes_') and not np.array_equal(clf.classes_, CLASSES):
            reorder_idx = [list(clf.classes_).index(c) for c in CLASSES]
            P = P[:, reorder_idx]
        
        # Calcul EU pour classification 4 bacs
        EU = P @ U.T  # (n_samples, n_classes) @ (n_classes, n_classes) = (n_samples, n_classes)
        avg_utility = EU.max(axis=1).mean()
        
        # Calcul de l'incertitude pour EKNN
        uncertainty = 0.0
        if 'EKNN' in name:
            masses = clf.get_belief_masses(X_va_scaled)
            uncertainty = masses[:, -1].mean()
        
        results[name] = {
            'accuracy': accuracy_score(y_va, y_pred),
            'avg_expected_utility': avg_utility,
            'uncertainty': uncertainty,
            'name': name
        }
        
        print(f"{name:12} | Précision: {results[name]['accuracy']:.3f} | EU: {avg_utility:.3f}", end="")
        if uncertainty > 0:
            print(f" | Incertitude: {uncertainty:.3f}")
        else:
            print()
    
    return results

# Exécution
knn_results = evaluate_knn_variants()

# Affichage du meilleur modèle
best_knn = max([r for r in knn_results.values() if 'EKNN' not in r['name']], 
               key=lambda x: x['avg_expected_utility'])
best_eknn = max([r for r in knn_results.values() if 'EKNN' in r['name']], 
                key=lambda x: x['avg_expected_utility'])

print(f"\\n🏆 Meilleur k-NN: {best_knn['name']} (EU={best_knn['avg_expected_utility']:.3f})")
print(f"🔮 Meilleur EKNN: {best_eknn['name']} (EU={best_eknn['avg_expected_utility']:.3f})")
print(f"📈 Gain EKNN: {best_eknn['avg_expected_utility'] - best_knn['avg_expected_utility']:.4f}")

ÉVALUATION k-NN vs EKNN
k-NN (k=5)   | Précision: 0.898 | EU: 0.346
k-NN (k=10)  | Précision: 0.894 | EU: 0.309
k-NN (k=10)  | Précision: 0.894 | EU: 0.309
EKNN (k=5)   | Précision: 0.713 | EU: -0.121 | Incertitude: 0.984
EKNN (k=5)   | Précision: 0.713 | EU: -0.121 | Incertitude: 0.984
EKNN (k=10)  | Précision: 0.590 | EU: -0.109 | Incertitude: 0.978
\n🏆 Meilleur k-NN: k-NN (k=5) (EU=0.346)
🔮 Meilleur EKNN: EKNN (k=10) (EU=-0.109)
📈 Gain EKNN: -0.4550
EKNN (k=10)  | Précision: 0.590 | EU: -0.109 | Incertitude: 0.978
\n🏆 Meilleur k-NN: k-NN (k=5) (EU=0.346)
🔮 Meilleur EKNN: EKNN (k=10) (EU=-0.109)
📈 Gain EKNN: -0.4550


### Analyse des Résultats k-NN vs EKNN

In [ ]:
# Tableau comparatif des résultats
knn_comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Type': 'EKNN' if name.startswith('EKNN') else 'kNN',
        'k': int(name.split('(k=')[1].split(')')[0]),
        'Accuracy': res['accuracy'],
        'Avg_Expected_Utility': res['avg_expected_utility'],
        'Uncertainty': res['uncertainty']
    }
    for name, res in knn_results.items()
])

knn_comparison_df = knn_comparison_df.sort_values('Avg_Expected_Utility', ascending=False)

print("RÉSUMÉ COMPARATIF: k-NN vs EKNN")
print("=" * 50)
print(knn_comparison_df.round(4))

# Analyse par type
print("\nANALYSE PAR TYPE:")
print("-" * 30)
for model_type in ['kNN', 'EKNN']:
    subset = knn_comparison_df[knn_comparison_df['Type'] == model_type]
    print(f"\n{model_type}:")
    print(f"  Meilleure utilité: {subset['Avg_Expected_Utility'].max():.4f}")
    print(f"  Précision moyenne: {subset['Accuracy'].mean():.3f}")
    if model_type == 'EKNN':
        print(f"  Incertitude moyenne: {subset['Uncertainty'].mean():.3f}")

# Meilleur modèle de chaque type
best_knn = knn_comparison_df[knn_comparison_df['Type'] == 'kNN'].iloc[0]
best_eknn = knn_comparison_df[knn_comparison_df['Type'] == 'EKNN'].iloc[0]

print(f"\n🏆 Meilleur k-NN classique: {best_knn['Model']} (EU: {best_knn['Avg_Expected_Utility']:.4f})")
print(f"🏆 Meilleur EKNN: {best_eknn['Model']} (EU: {best_eknn['Avg_Expected_Utility']:.4f})")

RÉSUMÉ COMPARATIF: k-NN vs EKNN
         Model  Type   k  Accuracy  Avg_Expected_Utility  Uncertainty
0   k-NN (k=5)   kNN   5    0.8983                0.3459       0.0000
1  k-NN (k=10)   kNN  10    0.8937                0.3094       0.0000
3  EKNN (k=10)  EKNN  10    0.5902               -0.1091       0.9784
2   EKNN (k=5)  EKNN   5    0.7127               -0.1207       0.9842

ANALYSE PAR TYPE:
------------------------------

kNN:
  Meilleure utilité: 0.3459
  Précision moyenne: 0.896

EKNN:
  Meilleure utilité: -0.1091
  Précision moyenne: 0.651
  Incertitude moyenne: 0.981

🏆 Meilleur k-NN classique: k-NN (k=5) (EU: 0.3459)
🏆 Meilleur EKNN: EKNN (k=10) (EU: -0.1091)


## 12. Analyse des Différences entre k-NN et EKNN

### Task 3d: Exemples où k-NN classique diffère d'EKNN

In [ ]:
def analyze_classification_4bins(results, X_val, y_val, n_examples=10):
    """
    Analyse des résultats de classification en 4 bacs avec le classificateur EU
    """
    print("\\n" + "="*80)
    print("ANALYSE DE LA CLASSIFICATION EN 4 BACS")
    print("="*80)
    
    # Prendre le meilleur modèle selon l'utilité espérée
    best_model = max(results.items(), key=lambda x: x[1]['avg_expected_utility'])
    model_name, model_results = best_model
    
    print(f"\\nAnalyse basée sur le modèle: {model_name}")
    print(f"Utilité espérée moyenne: {model_results['avg_expected_utility']:.3f}")
    
    eu_clf = model_results['eu_classifier']
    classic_pred = model_results['classic_predictions']
    eu_pred = model_results['eu_predictions'] 
    utilities = model_results['utilities']
    
    # Calculer les probabilités
    P = eu_clf.predict_proba(X_val)
    
    # Créer un DataFrame pour l'analyse
    analysis_df = pd.DataFrame({
        'y_true': y_val,
        'classic_pred': classic_pred,
        'eu_pred': eu_pred,
        'P_ABS': P[:, 0],
        'P_HiPS': P[:, 1], 
        'P_PE': P[:, 2],
        'P_PP': P[:, 3],
        'EU_ABS': utilities[:, 0],
        'EU_HiPS': utilities[:, 1],
        'EU_PE': utilities[:, 2],
        'EU_PP': utilities[:, 3],
        'max_EU': utilities.max(axis=1),
        'confidence': P.max(axis=1)  # Confiance = probabilité maximale
    })
    
    print(f"\\nMatrice d'utilité utilisée (4x4):")
    U = eu_clf.utility_matrix
    print(f"{'Action \\\\ État':<15} {'ABS':<8} {'HiPS':<8} {'PE':<8} {'PP':<8}")
    for i, action in enumerate(['ABS', 'HiPS', 'PE', 'PP']):
        print(f"{'Classer comme ' + action:<15} {U[i,0]:<8.1f} {U[i,1]:<8.1f} {U[i,2]:<8.1f} {U[i,3]:<8.1f}")
    
    return analysis_df, model_name

def analyze_classification_behavior_4bins(analysis_df):
    """Analyse du comportement de classification en 4 bacs"""
    
    print("\\n1. PERFORMANCE PAR CLASSE")
    print("-" * 50)
    
    behavior_by_class = []
    for classe in CLASSES:
        mask = analysis_df['y_true'] == classe
        if mask.sum() > 0:
            # Taux de classification correcte
            correct_rate = (analysis_df.loc[mask, 'eu_pred'] == classe).mean()
            
            # Utilité espérée moyenne pour cette classe
            avg_eu = analysis_df.loc[mask, 'max_EU'].mean()
            
            # Confiance moyenne
            avg_confidence = analysis_df.loc[mask, 'confidence'].mean()
            
            # Classe la plus fréquemment prédite à tort
            wrong_preds = analysis_df.loc[mask & (analysis_df['eu_pred'] != classe), 'eu_pred']
            most_common_error = wrong_preds.mode()[0] if len(wrong_preds) > 0 else None
            
            behavior_by_class.append({
                'Classe': classe,
                'Taux_Correct': correct_rate,
                'Avg_EU': avg_eu,
                'Avg_Confidence': avg_confidence,
                'Erreur_Freq': most_common_error,
                'Nb_Echantillons': mask.sum()
            })
            
            print(f"{classe}: {correct_rate:.1%} correct ({mask.sum()} échantillons)")
            print(f"  EU moyenne: {avg_eu:.3f}, Confiance: {avg_confidence:.3f}")
            if most_common_error:
                print(f"  Erreur fréquente: classé comme {most_common_error}")

    behavior_df = pd.DataFrame(behavior_by_class)
    
    print("\\n2. MATRICE DE CONFUSION EU vs VRAIE CLASSE")
    print("-" * 50)
    confusion_eu = pd.crosstab(analysis_df['y_true'], analysis_df['eu_pred'], 
                              margins=True, normalize='index')
    print(confusion_eu.round(3))
    
    print("\\n3. EXEMPLES DE DÉCISIONS EU vs CLASSIQUE")
    print("-" * 50)
    
    # Exemples où EU diffère du classificateur classique
    different_decisions = analysis_df['classic_pred'] != analysis_df['eu_pred']
    
    if different_decisions.sum() > 0:
        print(f"Différences EU vs Classique: {different_decisions.sum()}/{len(analysis_df)} ({different_decisions.mean():.1%})")
        
        examples = analysis_df[different_decisions].head(5)
        for idx, (_, row) in enumerate(examples.iterrows()):
            print(f"\\nExemple {idx+1}:")
            print(f"  Vraie classe: {row['y_true']}")
            print(f"  Classique: {row['classic_pred']} | EU: {row['eu_pred']}")
            print(f"  Probabilités: ABS={row['P_ABS']:.3f}, HiPS={row['P_HiPS']:.3f}, PE={row['P_PE']:.3f}, PP={row['P_PP']:.3f}")
            print(f"  Utilités EU: ABS={row['EU_ABS']:.3f}, HiPS={row['EU_HiPS']:.3f}, PE={row['EU_PE']:.3f}, PP={row['EU_PP']:.3f}")
            print(f"  → EU choisit classe avec meilleure utilité espérée")
    else:
        print("Aucune différence trouvée entre EU et classificateur classique.")
    
    return behavior_df

# Application de l'analyse pour les 4 bacs
print("Analyse en cours... (utilise les résultats du test précédent)")
print("Note: Pour une analyse complète, utiliser compare_classifiers_4bins avec tous les modèles")

COMPARAISON k-NN vs EKNN
Meilleur k-NN: k-NN (k=5) (EU=0.3459)
Meilleur EKNN: EKNN (k=10) (EU=-0.1091)
\nAmélioration EU: -0.4550
Incertitude moyenne EKNN: 0.978
\nComparaison des performances:
  Précision k-NN: 0.898
  Précision EKNN: 0.590
  Différence précision: -0.308
\nEstimation des améliorations individuelles:
Échantillons potentiellement améliorés: ~0 (0.0%)
Échantillons potentiellement dégradés: ~3077 (100.0%)
